In [ ]:
from pyscf import gto, dft
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import os

ROOT_DIR = os.path.dirname(os.path.abspath("__file__"))

xyz_file = "active_site_202604222145.xyz"

with open(xyz_file, "r") as f:
    lines = f.readlines()

# Remove XYZ header
atom_block = "".join(lines[2:])

# uncomment and use single Zn atom for testing
#atom_block = "Zn 0 0 0"

# Timestamp for output files
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Output filenames
log_file = Path(f"{ROOT_DIR}/logs/pyscf_scf_log_{timestamp}.txt")
plot_file = Path(f"{ROOT_DIR}/plots/pyscf_scf_plot_{timestamp}.png")

mol = gto.M(
    atom=atom_block,
    basis="lanl2dz",
    charge=2,
    spin=0,
    unit="Ang",
    verbose=4
)

# Closed-shell -> RKS is the better first choice
mf = dft.RKS(mol)
mf.xc = "B3LYP"
mf.max_cycle = 100
mf.level_shift = 0.2

# Store SCF energies
iterations = []
energies = []

def scf_callback(envs):
    cycle = envs["cycle"]
    e_tot = envs["e_tot"]
    iterations.append(cycle)
    energies.append(e_tot)

mf.callback = scf_callback

# Run SCF
energy = mf.kernel()

print("\nFinal electronic energy (Hartree):", energy)
print("Converged:", mf.converged)
print("Number of electrons:", mol.nelectron)

# Save text log
with open(log_file, "w") as f:
    f.write("PySCF SCF Run Log\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"XYZ source file: {xyz_file}\n")
    f.write(f"Method: RKS\n")
    f.write(f"Functional: {mf.xc}\n")
    f.write(f"Basis: def2-SVP\n")
    f.write(f"Charge: {mol.charge}\n")
    f.write(f"Spin: {mol.spin}\n")
    f.write(f"Converged: {mf.converged}\n")
    f.write(f"Final electronic energy (Hartree): {energy}\n")
    f.write(f"Number of electrons: {mol.nelectron}\n")
    f.write("\nAtoms used in simulation:\n")
    f.write(atom_block)
    f.write("\nSCF Iteration Energies:\n")
    f.write("Iteration\tEnergy_Hartree\n")
    for i, e in zip(iterations, energies):
        f.write(f"{i}\t{e:.12f}\n")

print(f"Saved SCF log to: {log_file}")

# Plot convergence
plt.figure()
plt.plot(iterations, energies, marker="o")
plt.xlabel("Iteration")
plt.ylabel("Energy (Hartree)")
plt.title("SCF Convergence")
plt.grid(True)
plt.tight_layout()
plt.savefig(plot_file, dpi=300)
plt.show()

print(f"Saved convergence plot to: {plot_file}")

System: uname_result(system='Darwin', node='Kaylas-Air', release='23.6.0', version='Darwin Kernel Version 23.6.0: Fri Jul  5 18:01:46 PDT 2024; root:xnu-10063.141.1~2/RELEASE_ARM64_T8112', machine='arm64')  Threads 1
Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ]
numpy 1.26.4  scipy 1.13.1  h5py 3.11.0
Date: Wed Apr 22 22:00:30 2026
PySCF version 2.8.0
PySCF path  /opt/homebrew/anaconda3/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 25
[INPUT] num. electrons = 140
[INPUT] charge = 2
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = Ang
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 C     -8.215000000000  -0.694000000000  11.586000000000 AA  -15.524100113302  -1.311469930448  21.894366879211 Bohr   0.0
[INPUT]  2 C     -7.589000000000  -1.5150